In [24]:
import pandas as pd
import numpy as np
import ast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. CARREGAMENTO E LIMPEZA DOS DADOS ---
print("Lendo datasets...")
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv')

# Merge
df = movies.merge(credits, on='title')
df = df[['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Funções de extração
def get_list(obj):
    try:
        return [i['name'] for i in ast.literal_eval(obj)]
    except: return []

def get_director(obj):
    try:
        for i in ast.literal_eval(obj):
            if i['job'] == 'Director':
                return [i['name']]
        return []
    except: return []

print("Limpando metadados...")
df['genres'] = df['genres'].apply(get_list)
df['keywords'] = df['keywords'].apply(get_list)
df['cast'] = df['cast'].apply(lambda x: get_list(x)[:3])
df['director'] = df['crew'].apply(get_director)
df['overview'] = df['overview'].fillna('')

Lendo datasets...
Limpando metadados...


In [34]:
# --- 2. CRIAÇÃO DA SOPA COM PESOS (BOOSTING) ---
# Aqui a gente força o score pra cima repetindo os termos importantes
def create_weighted_soup(x):
    # Diretor (Peso 3), Keywords e Gêneros (Peso 2), Elenco e Sinopse (Peso 1)
    director = " ".join(x['director'] * 3)
    genres = " ".join(x['genres'] * 2)
    keywords = " ".join(x['keywords'] * 2)
    cast = " ".join(x['cast']*1)
    overview =  " ".join(x['overview']*1)
    return f"{director} {genres} {keywords} {cast} {overview}"

df['weighted_soup'] = df.apply(create_weighted_soup, axis=1)

In [35]:
# --- 3. MODELAGEM DEEP LEARNING (MODELO ROBUSTO) ---
print("Carregando all-mpnet-base-v2 (Isso pode demorar um pouco)...")
# Este modelo é superior ao MiniLM em capturar nuances
model = SentenceTransformer('all-mpnet-base-v2')

print("Gerando Embeddings de alta fidelidade...")
movie_embeddings = model.encode(df['weighted_soup'].tolist(), show_progress_bar=True)
similarity = cosine_similarity(movie_embeddings)

Carregando all-mpnet-base-v2 (Isso pode demorar um pouco)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Gerando Embeddings de alta fidelidade...


Batches:   0%|          | 0/151 [00:00<?, ?it/s]

In [36]:
# --- 4. FUNÇÃO DE TESTE E ANÁLISE FILTRADA ---
def testar_modelo_final(filme_alvo, generos_usuario, top_n=10):
    try:
        idx = df[df['title'] == filme_alvo].index[0]
    except:
        return f"Filme '{filme_alvo}' não encontrado."

    # Pega os 100 mais similares para filtrar depois
    distances = sorted(list(enumerate(similarity[idx])), reverse=True, key=lambda x: x[1])

    resultados = []
    for i in distances[1:]:
        m_idx = i[0]
        m_genres = df.iloc[m_idx].genres
        m_title = df.iloc[m_idx].title
        m_score = i[1]

        # Filtro: deve ter ao menos um dos gêneros escolhidos
        match_generos = list(set(m_genres) & set(generos_usuario))

        if match_generos:
            resultados.append({
                'Título': m_title,
                'Gêneros': m_genres,
                'Gênero_Match': match_generos[0],
                'Score_Semantico': round(m_score, 4)
            })

        if len(resultados) == top_n:
            break

    df_res = pd.DataFrame(resultados)
    return df_res.sort_values(by=['Score_Semantico'], ascending=False)

In [37]:
# --- 5. EXECUÇÃO DO TESTE ---
target = 'Avatar'
preferencias = ['Action', 'Adventure', 'Thriller', 'Romance']

print(f"\n--- TESTE DE ALTA PRECISÃO PARA: {target} ---")
resultado_final = testar_modelo_final(target, preferencias)

print("\n")
print(resultado_final.to_markdown(index=False))


--- TESTE DE ALTA PRECISÃO PARA: Avatar ---


| Título                              | Gêneros                                                | Gênero_Match   |   Score_Semantico |
|:------------------------------------|:-------------------------------------------------------|:---------------|------------------:|
| The Abyss                           | ['Adventure', 'Action', 'Thriller', 'Science Fiction'] | Thriller       |            0.8933 |
| Aliens                              | ['Horror', 'Action', 'Thriller', 'Science Fiction']    | Thriller       |            0.8634 |
| The Terminator                      | ['Action', 'Thriller', 'Science Fiction']              | Thriller       |            0.8611 |
| Independence Day: Resurgence        | ['Action', 'Adventure', 'Science Fiction']             | Adventure      |            0.8441 |
| Terminator 2: Judgment Day          | ['Action', 'Thriller', 'Science Fiction']              | Thriller       |            0.8428 |
| Independence 

In [38]:
import pickle

# Salva o DataFrame processado (com a coluna de IDs para buscar posters)
pickle.dump(df, open('movies_list.pkl', 'wb'))

# Salva a matriz de similaridade (o "cérebro" do sistema)
pickle.dump(similarity, open('similarity_matrix.pkl', 'wb'))

print("Arquivos 'movies_list.pkl' e 'similarity_matrix.pkl' salvos com sucesso!")

Arquivos 'movies_list.pkl' e 'similarity_matrix.pkl' salvos com sucesso!


In [39]:
from google.colab import files

files.download('movies_list.pkl')
files.download('similarity_matrix.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

O que foi feito? (Explicação do Projeto)
Se você precisar apresentar esse projeto ou colocar no seu portfólio, aqui está o resumo técnico:

Objetivo: Criar um sistema de recomendação de filmes capaz de entender o contexto semântico das histórias, indo além de simples palavras-chave.

Metodologia:

Engenharia de Dados (Metadata Soup): Criamos uma "sopa de metadados" enriquecida, unindo Sinopse, Diretor (com peso 3x), Elenco e Palavras-chave. Isso garante que o modelo entenda não só a história, mas também o estilo do diretor e os subgêneros.

Deep Learning (Embeddings): Utilizamos o modelo all-mpnet-base-v2 (baseado em Transformers/BERT). Ele transforma o texto em vetores de 768 dimensões. Filmes com temas similares ocupam espaços próximos nesse universo matemático.

Similaridade do Cosseno: Calculamos a distância angular entre os vetores. Quanto maior o cosseno (mais próximo de 1.0), mais parecidos os filmes são.

Sistema Híbrido: O motor de busca é movido pela IA (Deep Learning), mas a saída é refinada por filtros de metadados (Gêneros), permitindo que o usuário controle o que quer ver dentro daquela "vibe" sugerida.

Resultados:
O modelo atingiu scores de confiança superiores a 80%, conseguindo identificar conexões complexas entre obras (ex: recomendando The Abyss para quem gosta de Avatar devido à direção de James Cameron e temática subaquática/alienígena).

Como rodar seu app:

Garanta que os arquivos .pkl estão na mesma pasta.

No terminal: streamlit run app.py